In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, Dataset
import os 
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
print(f"GPU available: {device if torch.cuda.is_available() else 'NO' }")

In [ ]:
data = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")

In [ ]:
"Splitting dataset into train and test data"
from sklearn.model_selection import train_test_split

train_df , test_df = train_test_split(data,train_size=0.8,random_state=42)
train_df = train_df.to_numpy()
test_df=test_df.to_numpy()

y_train , x_train= train_df[:,0] ,train_df[:,1:].astype(np.float32)
y_test, x_test = test_df[:, 0], test_df[:, 1:].astype(np.float32)

In [ ]:
x_train, y_train, x_test, y_test = map(torch.tensor, (x_train, y_train, x_test, y_test))
x_train, y_train, x_test, y_test = [t.to(device) for t in [x_train, y_train, x_test, y_test]]

x_train/=255.0
x_test/=255.0
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(x_train, y_train)
test_ds  = TensorDataset(x_test,  y_test)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
"Writing Helper Functions"
epsilon=1e-6
def relu(x):
    return x.clamp(min=0)

def batch_norm(x, gamma) :
    batch_mean = x.mean(dim=0)
    batch_var = (x).var(dim=0)
    x_norm = (x - batch_mean) / torch.sqrt(batch_var + eps)
    return x_norm

"give x tensor"
def softmax(x):
    y = torch.exp(x)
    den_sum= torch.sum(y, dim=1)
    return y/sum

In [ ]:
class Regression():
    def __init__(self , inp_shape, out_shape):
        self.w = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((out_shape, inp_shape)).to(device).requires_grad_()
        self.b = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((out_shape, )).to(device).requires_grad_()

    def __call__(self,x):
        return x@self.w.t()+self.b

In [ ]:
class Neural_Network():
    def __init__(self, hidden):
        self.hidden_features=hidden
        self.lr1=Regression(784,self.hidden_features)
        self.lr2=Regression(self.hidden_features,self.hidden_features)
        self.lr3=Regression(self.hidden_features,10)
        self.params = [self.lr1.w, self.lr2.w, self.lr3.w, self.lr1.b, self.lr2.b, self.lr3.b]

    def __call__(self,x):
        input=x
        y1 = relu(self.lr1(input))  #Using the call function inside Regression Class"
        y2= relu(self.lr2(y1))
        y3= self.lr3(y2)
        return y3

In [ ]:
def cross_entropy(logits, y):
    logprob = torch.log_softmax(logits, dim=1)

    loss = - logprob[torch.arange(len(y)),y].mean()
    
    return loss

In [ ]:
BATCH_SIZE = 64
epochs = 200
Epoch_eval = 5
lr = 1e-3
hidden = 512

In [ ]:
def accuracy(model, data) :
    acc = 0
    c = 0
    for x, y in data :
        ypred = model(x)
        yhat = torch.argmax(ypred, -1)
        acc += (y == yhat).to(float).mean()
        c+=1
    acc = acc / c
    return acc

In [ ]:
MNIST = Neural_Network(hidden)

for i in range(epochs):
    for x,y in train_dl:
        y_pred=MNIST(x)

        loss = cross_entropy(y_pred,y)

        loss.backward() # This calculates the gradient for backward propagation

        with torch.no_grad():
            for j in range(len(MNIST.params)):
                MNIST.params[j]-= lr*MNIST.params[j].grad # THe gradient Descent Step

        for j in range(len(MNIST.params)):
            MNIST.params[j].grad.zero_()

    if (i+1)%Epoch_eval==0:
        print(f"Epoch {i+1}, Loss = {loss} , Accuracy = {accuracy(MNIST,test_dl)}")